In [110]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
random_state = 45
target = 'saleprice'

**1---Data Inspection**

In [123]:
df_raw = pd.read_csv('data/train.csv')
print('---raw data---')
display(df_raw.head())
print('---raw data shape---')
display(df_raw.shape)
print('---data info---')
display(df_raw.info())
print('---data description---')
display(df_raw.describe().T)


---raw data---


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


---raw data shape---


(1460, 81)

---data info---
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  Ove

None

---data description---


,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.610009,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,1460.0,56.897260,42.300571,20.0,20.00,50.0,70.00,190.0
LotFrontage,1201.0,70.049958,24.284752,21.0,59.00,69.0,80.00,313.0
LotArea,1460.0,10516.828082,9981.264932,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,1460.0,6.099315,1.382997,1.0,5.00,6.0,7.00,10.0
OverallCond,1460.0,5.575342,1.112799,1.0,5.00,5.0,6.00,9.0
YearBuilt,1460.0,1971.267808,30.202904,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1460.0,1984.865753,20.645407,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,1452.0,103.685262,181.066207,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,1460.0,443.639726,456.098091,0.0,0.00,383.5,712.25,5644.0


**2--Renaming the columns for uniformatiy**

In [124]:
df_raw = df_raw.rename(columns = {i:i.lower() for i in df_raw.columns})
df_raw.head()
df = df_raw.copy()
df= df.drop(columns=['id'])


**Changing the type of features based on the info they provide**

In [113]:
print('--mssubclass represents the class of sold property in real-state--\n but here its represented as an integer\n --must be transfered into nominal category, as it represents category of sold property based on style, architecture,..--')
display(df['mssubclass'].unique())
print('--mosold represents the month when the property was sold--\n which is represented as an integer\n --must be transfered into nominal category, as it represents category of month--')
display(df['mosold'].unique())
df['mssubclass']=df['mssubclass'].astype(str)
df['mosold']=df['mosold'].astype(str)

--mssubclass represents the class of sold property in real-state--
 but here its represented as an integer
 --must be transfered into nominal category, as it represents category of sold property based on style, architecture,..--


array([ 60,  20,  70,  50, 190,  45,  90, 120,  30,  85,  80, 160,  75,
       180,  40])

--mosold represents the month when the property was sold--
 which is represented as an integer
 --must be transfered into nominal category, as it represents category of month--


array([ 2,  5,  9, 12, 10,  8, 11,  4,  1,  7,  3,  6])

**1. Classification of datas based on datatypes**

In [143]:
def classify(df):
    target = 'saleprice'
    numerical = [feature for feature in df.select_dtypes(include=np.number).columns if feature!=target]
    categorical = [feature for feature in df.select_dtypes(exclude=np.number).columns if feature!=target]
    continuous = [feature for feature in numerical if df[feature].nunique() >20 if feature!=target] 
    discrete = [feature for feature in numerical if df[feature].nunique() <=20 if feature!=target]
    return numerical, categorical, continuous, discrete
numerical, categorical, continuous, discrete = classify(df)   
print(f'number of numerical features: {len(numerical)}')
print(f'number of categorical features: {len(categorical)}')
print(f'number of continuous features: {len(continuous)}')
print(f'number of discrete features: {len(discrete)}')

number of numerical features: 45
number of categorical features: 42
number of continuous features: 23
number of discrete features: 22


**2. Checking the null values and duplicate values**

In [115]:
data_report = pd.DataFrame(
    {
        'null_count':df.isna().sum(),  #counting the total number of null values in each feature
        'null_percentage':(df.isna().mean() * 100).round(2)  #and converting them into percentage based on whole training examples
    }
).query('null_count>0').sort_values('null_percentage',ascending=False) #only selecting the datas having null values greater than 0 and sorting them in descending order based on null percentage
print('---The report on null values and null percentage based on every features having null values---')
display(data_report)
print(f'The features having duplicate values: {df.duplicated().sum()}')

---The report on null values and null percentage based on every features having null values---


,null_count,null_percentage
poolqc,1453,99.52
miscfeature,1406,96.30
alley,1369,93.77
fence,1179,80.75
masvnrtype,872,59.73
fireplacequ,690,47.26
lotfrontage,259,17.74
garagetype,81,5.55
garageyrblt,81,5.55
garagefinish,81,5.55


The features having duplicate values: 0


**2.1 Filling null values**

**In this ames house_price prediction datasets, most of the null values in the dataset represents that the house doesnot has this particular feature, so rather than dropping the null values feature or filling them using median or mode, we fill them with None or 0 based on the type of feature, to let the model know that the house doesnot has this particular feature**

In [116]:
none_features = ['poolqc', 'miscfeature', 'alley', 'fence', 'masvnrtype',
                 'fireplacequ', 'garagetype', 'garagefinish', 'garagequal',
                 'garagecond', 'bsmtexposure', 'bsmtfintype2', 'bsmtqual', 
                 'bsmtcond', 'bsmtfintype1']
zero_features = ['masvnrarea', 'garageyrblt']
for feature in none_features:
    df[feature] = df[feature].fillna('None')
for feature in zero_features:
    df[feature] = df[feature].fillna(0)
print('After filling the null values accordingly, the remaining features with null values: \n', df.isna().sum()[df.isna().sum()>0]) 


After filling the null values accordingly, the remaining features with null values: 
 lotfrontage    259
electrical       1
dtype: int64


**2.2 Every house built has a lotfrontage, so null value of this feature is actually a dataentry mistake and, as the lotfrontage of the houses in the same neighbourhood is almost identical , so null values in this feature can be filled using the median based on the neighbourhood's lotfrontage**

In [117]:
df['lotfrontage'] =df.groupby('neighborhood')['lotfrontage'].transform(lambda x:x.fillna(x.median())) #filling the null values of the lotfrontage using the median of the lotfrontage based on the group


**2.3 Every house built has electricity facility, so null values in them is actually data entry default, so we must fill them based on freqeuntly occuring value**

In [118]:
df['electrical'] = df['electrical'].fillna(df['electrical'].mode()[0])
df.isnull().sum()[df.isnull().sum()>0]

Series([], dtype: int64)

**3.Initial Feature Selection**

**3.1 Checking the constant features or near constant features** 

In [119]:
vt = VarianceThreshold(0.01)
vt.fit(df[numerical])
variance_result = vt.get_support()
constant_num_features = [col for col,s in zip(df[numerical],variance_result) if not s]
print(f'The numerical features having constant values or near constant values: {constant_num_features}')


The numerical features having constant values or near constant values: []


In [120]:
constant_cat_features = [feature for feature in categorical if df[feature].value_counts(normalize=True).iloc[0]>0.99]
mi_storage = {}
for feature in constant_cat_features:
    x = df[feature].astype('category').cat.codes.to_frame(name=feature) 
    y = df['saleprice']
    mi = mutual_info_regression(x,y)
    mi_storage[feature] = mi
mi_storage
    

{'street': array([0]),
 'utilities': array([0.00358717]),
 'poolqc': array([0.00966535])}

**Here as we can see, utilities is quasi_constant feature and also has very weak mutual info regression with the saleprice, so its valid to drop utilities feature**

In [125]:
df = df.drop(columns = ['utilities'])  

In [126]:
df.columns

Index(['mssubclass', 'mszoning', 'lotfrontage', 'lotarea', 'street', 'alley',
       'lotshape', 'landcontour', 'lotconfig', 'landslope', 'neighborhood',
       'condition1', 'condition2', 'bldgtype', 'housestyle', 'overallqual',
       'overallcond', 'yearbuilt', 'yearremodadd', 'roofstyle', 'roofmatl',
       'exterior1st', 'exterior2nd', 'masvnrtype', 'masvnrarea', 'exterqual',
       'extercond', 'foundation', 'bsmtqual', 'bsmtcond', 'bsmtexposure',
       'bsmtfintype1', 'bsmtfinsf1', 'bsmtfintype2', 'bsmtfinsf2', 'bsmtunfsf',
       'totalbsmtsf', 'heating', 'heatingqc', 'centralair', 'electrical',
       '1stflrsf', '2ndflrsf', 'lowqualfinsf', 'grlivarea', 'bsmtfullbath',
       'bsmthalfbath', 'fullbath', 'halfbath', 'bedroomabvgr', 'kitchenabvgr',
       'kitchenqual', 'totrmsabvgrd', 'functional', 'fireplaces',
       'fireplacequ', 'garagetype', 'garageyrblt', 'garagefinish',
       'garagecars', 'garagearea', 'garagequal', 'garagecond', 'paveddrive',
       'wooddecksf', 'o

**4. Initial Feature Engineering**


**4.1 Creating new features**

In [135]:
df['house_age'] = df['yrsold'] - df['yearbuilt']  #creating the age of house
df['garage_age'] = np.where(df['garageyrblt']!=0,df['yrsold'] - df['garageyrblt'],-1) #creating garage_age
df['total_sf'] = df['totalbsmtsf'] + df['1stflrsf'] + df['2ndflrsf'] #calcualting the total square footage
df['total_baths'] = df['fullbath'] + (df['halfbath'] * 0.5) + df['bsmtfullbath'] + (df['bsmthalfbath'] * 0.5) #calculating the Total bathrooms
df['total_porch_sf'] = df['openporchsf'] + df['enclosedporch'] + df['3ssnporch'] + df['screenporch'] #calculating the total porch area
df['total_outdoor_sf'] = df['wooddecksf'] + df['total_porch_sf'] #calcualting the total outdoor space
df['remodeled_age'] = df['yrsold'] - df['yearremodadd'] #calculating the age of house since it was remodeled
df['has_pool'] = (df['poolarea'] > 0).astype(int) #Has pool?
df['has_garage'] = (df['garagearea'] > 0).astype(int) #Has garage?
df['has_fireplace'] = (df['fireplaces'] > 0).astype(int) #Has fireplace?
df['has_basement'] = (df['totalbsmtsf'] > 0).astype(int)# Has basement?
df['has_2nd_floor'] = (df['2ndflrsf'] > 0).astype(int) #Has 2nd floor?
df['has_masonry'] = (df['masvnrarea'] > 0).astype(int) #Has masonry veneer?



**Dropping the date features, after the creation of related age features**

In [136]:
df = df.drop(columns=['yearbuilt','yrsold','garageyrblt','yearremodadd'])


In [139]:
numerical, categorical, continuous, discrete = classify(df)   #again classifying the datasets after addition of new features

**5. Feature Selection** 

**5.1 Checking correlation of numerical features with the target variable, and extracting the important features**

In [150]:
target = 'saleprice'
corr_report = []
for feature in numerical:
    corr = df[feature].corr(df[target])  #finding the correlation of all the numeric features with the target variable
    corr_report.append({
        'feature':feature,
        'corr_with_target':corr
    })
df_cor_report = pd.DataFrame(corr_report)
imp_features = df_cor_report[df_cor_report['corr_with_target'].abs()>0.1].sort_values('corr_with_target',ascending=False,key = lambda x:x.abs())
imp_features


,feature,corr_with_target
3,overallqual,0.790982
34,total_sf,0.782260
13,grlivarea,0.708624
22,garagecars,0.640409
35,total_baths,0.631731
23,garagearea,0.623431
9,totalbsmtsf,0.613581
10,1stflrsf,0.605852
16,fullbath,0.560664
20,totrmsabvgrd,0.533723


**5.2 Checking multi_collinearity of important numerical features**